# Cycle Matching Experiments

We test the snarl score on point clouds sampled uniformly on cubes.
Running the code requires Ripser in the version from https://github.com/inesgare/interval-matching/

**Required packages and versions:**
 - Numpy: 1.26.4
 - Ripser: 0.6.14
 - Scipy: 1.17.1
 - Pandas: 3.0.3
 - Matplotlib: 3.10.9
 - Guhdi: 3.11.0
 - Joblib: 1.5.3

In [8]:
import os
import sys
import numpy as np

from match.utils_PH import *
from match.utils_plot import *
from match.utils_data import *
from ripser import ripser
from scipy.spatial.distance import cdist
from collections import defaultdict

module_path = os.path.abspath(os.path.join(''))
if module_path not in sys.path:
    sys.path.append(module_path)


## Required Functions

In [9]:
def compute_dist_mat(data):
    distances = np.sqrt( np.sum( (data[:, None, :] - data[None, :, :])**2, axis=-1) )
    distances /= distances.max()
    return distances

def cycle_length(pts, rep):
    """Computes the euclideanlength of a cycle."""
    rep = np.array(rep)
    p1 = pts[rep[:, 0]]
    p2 = pts[rep[:, 1]]
    return np.sum(np.linalg.norm(p2 - p1, axis=1))

def rep_to_order(rep):
    """
    Converts a representation of a cycle (list of edges) into an ordered list of vertices.
    """
    neighbors = defaultdict(list)
    for e in rep:
        u, v = int(e[0]), int(e[1])
        neighbors[u].append(v)
        neighbors[v].append(u)

    start = int(rep[0][0])
    order = [start]
    visited_edges = set()
    current = start

    while True:
        next_node = None
        for nb in neighbors[current]:
            edge = (min(current, nb), max(current, nb))
            if edge not in visited_edges:
                next_node = nb
                visited_edges.add(edge)
                break

        if next_node is None or next_node == start:
            break

        order.append(next_node)
        current = next_node

    return order

def snarl_score(match_idx):
    """
    Computes the snarl score for a given match index.
    """
    a, b = matched_X_Y[match_idx]
    rep_L1 = tight_reps_X[1][a]
    rep_L2 = tight_reps_Y[1][b]

    order_L1 = rep_to_order(rep_L1)
    order_L2 = rep_to_order(rep_L2)

    rep_L2_from_L1 = [
        [order_L1[i], order_L1[(i + 1) % len(order_L1)]]
        for i in range(len(order_L1))
    ]
    rep_L1_from_L2 = [
        [order_L2[i], order_L2[(i + 1) % len(order_L2)]]
        for i in range(len(order_L2))
    ]

    len_L1_original  = cycle_length(layer_1, rep_L1)
    len_L2_original  = cycle_length(layer_2, rep_L2)
    len_L2_from_L1   = cycle_length(layer_2, rep_L2_from_L1)
    len_L1_from_L2   = cycle_length(layer_1, rep_L1_from_L2)

    ratio_1 = len_L2_from_L1 / len_L2_original   
    ratio_2 = len_L1_from_L2 / len_L1_original 

    geo_mean = np.sqrt(ratio_1 * ratio_2)

    return geo_mean

## Generating the Point Clouds

The number of points and dimensions can be chosen

In [10]:
N_POINTS = 500
N_DIMS = 5

np.random.seed(42)
layer_1 = random_cloud = np.random.uniform(low=-1.0, high=1.0, size=(N_POINTS, N_DIMS)).astype(np.float32)

np.random.seed(43)
layer_2 = random_cloud = np.random.uniform(low=-1.0, high=1.0, size=(N_POINTS, N_DIMS)).astype(np.float32)

distances_1 = np.sqrt( np.sum( (layer_1[:, None, :] - layer_1[None, :, :])**2, axis=-1) )
distances_2 = np.sqrt( np.sum( (layer_2[:, None, :] - layer_2[None, :, :])**2, axis=-1) )

distances_1 /= distances_1.max()
distances_2 /= distances_2.max()

matched_X_Y, affinity_X_Y, bars_reps_X, bars_reps_Y = matching_from_distmat(distances_1, distances_2)

bars_X, reps_X, tight_reps_X = bars_reps_X
bars_Y, reps_Y, tight_reps_Y = bars_reps_Y

print(len(matched_X_Y))

reducing column 363/406 (diameter 0.23279)

bars_X: {0: [], 1: [[0.127113, 0.129835], [0.131202, 0.136468], [0.142062, 0.145475], [0.146931, 0.148218], [0.151853, 0.152708], [0.152265, 0.154332], [0.143123, 0.156261], [0.144394, 0.15708], [0.149777, 0.157568], [0.14619, 0.158411], [0.151262, 0.158996], [0.157397, 0.159272], [0.156113, 0.159778], [0.146399, 0.160137], [0.14786, 0.160207], [0.159004, 0.160645], [0.160605, 0.161921], [0.146047, 0.163607], [0.16309, 0.16394], [0.162064, 0.16394], [0.158223, 0.164069], [0.158363, 0.165105], [0.164888, 0.165387], [0.160132, 0.165387], [0.147983, 0.167728], [0.158215, 0.168417], [0.153437, 0.168734], [0.145467, 0.168992], [0.163603, 0.169315], [0.160219, 0.169602], [0.16179, 0.170193], [0.159769, 0.170265], [0.161226, 0.171033], [0.165179, 0.171424], [0.170288, 0.171467], [0.139052, 0.171752], [0.148452, 0.172353], [0.166697, 0.172923], [0.144653, 0.173514], [0.173084, 0.173896], [0.164487, 0.1742], [0.162128, 0.174794], [0.160109, 0.17538], [0.173568, 0.175634], [0.151987, 0.175777], 

reducing column 369/405 (diameter 0.233955)

## Evaluation

For all matches we compute and output the snarl scores and the edges of the cycles in the two layers.

In [11]:
# Computing the snarl scores and edge counts for each match
results = []
for match_idx in range(len(matched_X_Y)):
    score = snarl_score(match_idx)
    a, b = matched_X_Y[match_idx]
    n_edges_L1 = len(tight_reps_X[1][a])
    n_edges_L2 = len(tight_reps_Y[1][b])
    results.append((match_idx, score, n_edges_L1, n_edges_L2))

# Sorting the results by snarl score in descending order
results_sorted = sorted(results, key=lambda x: x[1], reverse=True)

# Calculating average edge counts
avg_edges_L1 = np.mean([r[2] for r in results])
avg_edges_L2 = np.mean([r[3] for r in results])
avg_edges_combined = np.mean([r[2] + r[3] for r in results]) / 2

# Printing the results
print("\n── Matches sorted by Snarl Score (descending) ──")
print(f"{'Rank':<6} {'Match-Idx':<12} {'Snarl Score':<14} {'Edges L1':<10} {'Edges L2':<10}")
print("-" * 52)
for rank, (match_idx, score, n_edges_L1, n_edges_L2) in enumerate(results_sorted, start=1):
    print(f"{rank:<6} {match_idx:<12} {score:<14.4f} {n_edges_L1:<10} {n_edges_L2:<10}")

print("-" * 52)
print(f"{'Ø':<6} {'':<12} {'':<14} {avg_edges_L1:<10.1f} {avg_edges_L2:<10.1f}")
print(f"  Ø Total edges (L1+L2)/2: {avg_edges_combined:.1f}")


── Matches sorted by Snarl Score (descending) ──
Rank   Match-Idx    Snarl Score    Edges L1   Edges L2  
----------------------------------------------------
1      18           3.8303         13         5         
2      23           3.8139         4          7         
3      15           3.8012         27         24        
4      2            3.7004         32         18        
5      31           3.6779         17         9         
6      4            3.6587         7          5         
7      13           3.6418         5          9         
8      30           3.6037         6          36        
9      29           3.5791         19         14        
10     22           3.4785         33         42        
11     10           3.4492         11         7         
12     9            3.4298         26         8         
13     7            3.3501         10         30        
14     5            3.3002         10         6         
15     1            3.1718         9      